In [4]:
import numpy as np
import pandas as pd

In [5]:
df=pd.read_csv('factory_sensor_simulator_2040.csv')

CURRENT_YEAR = 2026
df['Machine_Age_Years'] = CURRENT_YEAR - df['Installation_Year']
df['Daily_Operational_Hours'] = df['Operational_Hours'] / np.maximum(df['Machine_Age_Years'] * 365, 1)
df['Hours_Since_Last_Maintenance'] = df['Last_Maintenance_Days_Ago'] * df['Daily_Operational_Hours']

In [6]:
eps = 1e-5 
df['Vibration_per_kW'] = df['Vibration_mms'] / (df['Power_Consumption_kW'] + eps)
df['Sound_per_kW'] = df['Sound_dB'] / (df['Power_Consumption_kW'] + eps)
df['Thermal_Efficiency_Ratio'] = df['Temperature_C'] / (df['Power_Consumption_kW'] + eps)
df['Fluid_Min_Level_pct'] = df[['Oil_Level_pct', 'Coolant_Level_pct']].min(axis=1)
df['Fluid_Average_pct'] = df[['Oil_Level_pct', 'Coolant_Level_pct']].mean(axis=1)

In [7]:
df['Mean_Hours_Between_Maintenance'] = df['Operational_Hours'] / (df['Maintenance_History_Count'] + 1)
df['Failure_to_Maintenance_Ratio'] = df['Failure_History_Count'] / (df['Maintenance_History_Count'] + 1)
df['Failure_Rate_per_1k_Hours'] = (df['Failure_History_Count'] / (df['Operational_Hours'] + 1)) * 1000

In [ ]:
df['Daily_Error_Rate_Last_30D'] = df['Error_Codes_Last_30_Days'] / 30.0
df['Errors_per_100_Hours'] = (df['Error_Codes_Last_30_Days'] / (df['Hours_Since_Last_Maintenance'] + 1)) * 100
df['AI_Override_Rate_per_1k_Hours'] = (df['AI_Override_Events'] / (df['Operational_Hours'] + 1)) * 1000

In [9]:
df['Temp_Dev_from_Type_Avg'] = df['Temperature_C'] - df.groupby('Machine_Type')['Temperature_C'].transform('mean')
df['Vibration_Dev_from_Type_Avg'] = df['Vibration_mms'] - df.groupby('Machine_Type')['Vibration_mms'].transform('mean')
df['Power_Dev_from_Type_Avg'] = df['Power_Consumption_kW'] - df.groupby('Machine_Type')['Power_Consumption_kW'].transform('mean')

In [10]:
df['Flag_Critical_Fluid'] = np.where((df['Oil_Level_pct'] < 20) | (df['Coolant_Level_pct'] < 20), 1, 0)
df['Flag_High_Stress'] = np.where((df['Temperature_C'] > 85) | (df['Vibration_mms'] > 5), 1, 0)
df['Flag_Overdue_Maintenance'] = np.where(df['Last_Maintenance_Days_Ago'] > 180, 1, 0)